# Topic Modelling example

## Libraries to add


In [1]:
from sklearn.feature_extraction.text import CountVectorizer
# Library for the Topic Modelling
from sklearn.decomposition import LatentDirichletAllocation

#NLTK library
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

#Pandas and Matplotlib library
import pandas as pd
import re
import string
import matplotlib.pyplot as pypl
import numpy as np

## Remove the contracted words

In [2]:
contract = {
"ain't": "is not",
"aren't": "are not",
"can't": "cannot",
"can't've": "cannot have",
"'cause": "because",
"could've": "could have",
"couldn't": "could not",
"couldn't've": "could not have",
"didn't": "did not",
"doesn't": "does not",
"don't": "do not",
"hadn't": "had not",
"hadn't've": "had not have",
"hasn't": "has not",
"haven't": "have not",
"he'd": "he would",
"he'd've": "he would have",
"he'll": "he will",
"he'll've": "he he will have",
"he's": "he is",
"how'd": "how did",
"how'd'y": "how do you",
"how'll": "how will",
"how's": "how is",
"I'd": "I would",
"I'd've": "I would have",
"I'll": "I will",
"I'll've": "I will have",
"I'm": "I am",
"I've": "I have",
"i'd": "i would",
"i'd've": "i would have",
"i'll": "i will",
"i'll've": "i will have",
"i'm": "i am",
"i've": "i have",
"isn't": "is not",
"it'd": "it would",
"it'd've": "it would have",
"it'll": "it will",
"it'll've": "it will have",
"it's": "it is",
"let's": "let us",
"ma'am": "madam",
"mayn't": "may not",
"might've": "might have",
"mightn't": "might not",
"mightn't've": "might not have",
"must've": "must have",
"mustn't": "must not",
"mustn't've": "must not have",
"needn't": "need not",
"needn't've": "need not have",
"o'clock": "of the clock",
"oughtn't": "ought not",
"oughtn't've": "ought not have",
"shan't": "shall not",
"sha'n't": "shall not",
"shan't've": "shall not have",
"she'd": "she would",
"she'd've": "she would have",
"she'll": "she will",
"she'll've": "she will have",
"she's": "she is",
"should've": "should have",
"shouldn't": "should not",
"shouldn't've": "should not have",
"so've": "so have",
"so's": "so as",
"that'd": "that would",
"that'd've": "that would have",
"that's": "that is",
"there'd": "there would",
"there'd've": "there would have",
"there's": "there is",
"they'd": "they would",
"they'd've": "they would have",
"they'll": "they will",
"they'll've": "they will have",
"they're": "they are",
"they've": "they have",
"to've": "to have",
"wasn't": "was not",
"we'd": "we would",
"we'd've": "we would have",
"we'll": "we will",
"we'll've": "we will have",
"we're": "we are",
"we've": "we have",
"weren't": "were not",
"what'll": "what will",
"what'll've": "what will have",
"what're": "what are",
"what's": "what is",
"what've": "what have",
"when's": "when is",
"when've": "when have",
"where'd": "where did",
"where's": "where is",
"where've": "where have",
"who'll": "who will",
"who'll've": "who will have",
"who's": "who is",
"who've": "who have",
"why's": "why is",
"why've": "why have",
"will've": "will have",
"won't": "will not",
"won't've": "will not have",
"would've": "would have",
"wouldn't": "would not",
"wouldn't've": "would not have",
"y'all": "you all",
"y'all'd": "you all would",
"y'all'd've": "you all would have",
"y'all're": "you all are",
"y'all've": "you all have",
"you'd": "you would",
"you'd've": "you would have",
"you'll": "you will",
"you'll've": "you will have",
"you're": "you are",
"you've": "you have"
}

## Function for removing the stop words and lemmatize the reviews 

In [3]:
stopword = set(stopwords.words('english'))
lemma = WordNetLemmatizer()

# Converting POS tag for lemmatizing according to grammatical context and position

def penntag(pen):
    morphy_tag = {'NN': 'n', 'JJ': 'a',
                  'VB': 'v', 'RB': 'r'}
    try:
        return morphy_tag[pen[:2]]
    except:
        return 'n'
 

In [4]:
def comment_cleaner(comm, comment_array):
    temp_comm = []
    megos = ' '
    
    # Uncontract the contracted words
    uncontracted = ' '.join([contract[word] if word in contract else word for word in comm.lower().split()])
    
    # Remove the stopwords
    stopwords_removed = [word for word in uncontracted.lower().split() if word not in stopword]
    
    # Add the POS tags to the converted text
    POS_words = nltk.pos_tag(stopwords_removed)
    
    # Lemmatize according to the POS tags
    for i in range(0, len(POS_words)):
        lemmas = lemma.lemmatize(POS_words[i][0], pos=penntag(POS_words[i][1]))
        temp_comm.append(lemmas)
        
    # Join the lemmatized words together
    megos = ' '.join(word for word in temp_comm)
    return megos

## Function for displaying the topics from the LDA Topic Modelling

In [8]:
topi = []

def print_topics(model, vectorizer, top_n=10):
    
    # Get the topics from the LDA Topic modelling
     for idx, topic in enumerate(model.components_):
         print("\nTopic : " ,str(idx),"\n")
        
        # Get the topics in descending order
         for i in topic.argsort()[:-top_n-1:-1]:
           topi.append((vectorizer.get_feature_names()[i]))
        
        # Display the topics 
         for i in range(0,top_n):
             print(topi[i])
        
        # Clear the 'topi' array
         topi.clear()

## Reading and cleaning the dataset

In [9]:

# Extract data from the dataset using pandas 
df123 = pd.read_csv('D:/Python/Machine Learning/Python Movie Dataset/movie_data.csv')

train_array = []

# Filter dataset based on sentiment (0 for negative, 1 for positive)
df123 = df123[df123['sentiment'] == 1]
    
# Lemmatize the review text 
df123['Review'] = df123['Review'].apply(lambda s: comment_cleaner(s, train_array))
    
# Remove the punctuation marks
df123['Review'] = df123['Review'].str.replace('[^\w\s]', ' ')
df123['Review'] = df123['Review'].str.replace('[\d+]', ' ')
df123['Review'] = df123['Review'].str.replace('(^| ).(( ).)*( |$)', ' ')

In [10]:
# Countvectorizer takes the frequency of the words as the weight and builds a vocabulary
tomol = CountVectorizer(stop_words=['br'],ngram_range=(1, 1))

# Fit and convert the review text using the Countvectorizer
tf = tomol.fit_transform(df123['Review'])

# Get the name of the words 
tf_feature = tomol.get_feature_names()

# Used for getting the topics using the LDA topic modelling (n_components is the number of topics) 
# Fit the transformed data to LDA using .fit()
lda = LatentDirichletAllocation(n_components=10).fit(tf)

# Display the topics using the print_topics() function  
print("LDA Model:")
print_topics(lda, tomol)
print("=" * 20)

LDA Model:

Topic :  0 

movie
one
film
like
make
people
see
it
time
life

Topic :  1 

show
series
episode
one
tv
get
like
first
character
good

Topic :  2 

love
one
play
character
the
like
woman
get
life
find

Topic :  3 

film
war
one
the
make
time
see
would
well
story

Topic :  4 

film
one
the
get
time
well
make
play
like
also

Topic :  5 

kelly
film
musical
one
dance
the
best
star
play
music

Topic :  6 

film
movie
story
one
character
well
see
make
great
the

Topic :  7 

film
one
horror
the
like
make
movie
get
see
scene

Topic :  8 

movie
see
like
it
good
one
watch
get
love
great

Topic :  9 

film
life
one
man
the
love
make
young
story
take
